#  OBJECTIVE

**This notebook implements the standalone prediction engine (`prediction_engine13`) for the AirFair-Vista platform** — a self-contained Python module that loads the serialised model, engineers features from raw user inputs, preprocesses them to the exact training format, makes a prediction, and returns a human-readable INR price with full unit test coverage.

> **Role:** Core prediction logic callable by Streamlit frontend, REST API, or batch scoring scripts | **Pattern:** Load → Engineer → Preprocess → Predict → Inverse-transform

In [ ]:
import joblib
import pandas as pd
import numpy as np

---
##  Step: Model Loading from Deployment Package

**Why:** The prediction engine loads the `deployment_package` dictionary (not just the raw model) — this ensures it also retrieves the validated `features` list that defines the exact column names and order the model expects. Loading both model and features as an atomic unit prevents the most common production bug: a model loaded from one file and features loaded from another, where a version mismatch silently produces wrong predictions without raising any error.

In [ ]:
#load model
BASE_PATH = "/content/drive/MyDrive/AirFair-Vista"

pipeline = joblib.load(
    f"{BASE_PATH}/models/flight_price_prediction_pipeline.pkl"
)

model = pipeline["model"]
features = pipeline["features"]

---
##  Step: Feature Engineering Function — `engineer_features(user_input)`

**Why:** Raw user inputs from the Streamlit UI (duration in hours, journey month, departure hour) must be transformed into the same derived features the model was trained on. This function reconstructs: `duration_minutes` = `duration_hours × 60`, temporal encoding, airline mean price lookup, and source frequency — the same pipeline used in Notebooks 2 and 13. Centralising this logic here means any feature engineering change only needs to be made in one place across the entire system.

In [ ]:
#create feature engineering function

def engineer_features(user_input):

    data = {}

    # Convert duration hours to minutes
    data["duration_hours"] = user_input["duration_hours"]
    data["duration_minutes"] = user_input["duration_hours"] * 60

    # Temporal features
    data["journey_month"] = user_input["journey_month"]
    data["journey_day"] = user_input["journey_day"]
    data["journey_weekday"] = user_input["journey_weekday"]

    # Departure hour
    data["dep_hour"] = user_input["dep_hour"]

    # Airline price influence
    data["Airline_mean_price"] = user_input["airline_mean_price"]

    # Route frequency
    data["Source_freq"] = user_input["source_freq"]

    return data

---
##  Step: Input Preprocessing — `preprocess_input(feature_dict)`

**Why:** `df.reindex(columns=features, fill_value=0)` is the critical safety operation that ensures: (1) columns appear in exactly the same order as during training — Random Forests use column index positions, not names, internally; (2) any feature the user didn't provide defaults to 0 rather than causing a `KeyError` crash; (3) extra columns the user accidentally sends are silently dropped. This makes the prediction engine robust to partial inputs and API schema changes.

In [ ]:
#preprocessing input
def preprocess_input(feature_dict):

    df = pd.DataFrame([feature_dict])

    # Align with training features
    df = df.reindex(columns=features, fill_value=0)

    return df

---
##  Step: Prediction Pipeline — `predict_flight_price(user_input)`

**Why:** The three-step pipeline (engineer → preprocess → predict) is wrapped in a single public function following the facade pattern. This is the single function the Streamlit frontend, REST API endpoint, or batch scoring script needs to call — they pass a dictionary of user inputs and receive a float price in INR. The `np.expm1()` inverse-transform converts the model's log-price output back to the actual rupee value the user sees, completing the end-to-end prediction cycle that began with `np.log1p()` in Notebook 1.

In [ ]:
#final pred function

def predict_flight_price(user_input):

    # Feature engineering
    engineered = engineer_features(user_input)

    # Preprocessing
    processed = preprocess_input(engineered)

    # Prediction
    prediction_log = model.predict(processed)

    # Convert back from log price
    price = np.expm1(prediction_log)

    return float(price[0])

In [ ]:
#test the function

if __name__ == "__main__":

    sample_user_input = {

        "duration_hours": 3,
        "journey_month": 5,
        "journey_day": 15,
        "journey_weekday": 2,
        "dep_hour": 10,
        "airline_mean_price": 9000,
        "source_freq": 1200

    }

    predicted_price = predict_flight_price(sample_user_input)

    print("Predicted Flight Price:", predicted_price)

Predicted Flight Price: 4390.453289608026


---
##  Step: Unit Tests — `run_unit_tests()`

**Why:** The unit tests validate the prediction engine across five edge cases: normal flight, very short flight (<1hr), long international-equivalent flight, peak season booking, and late-night departure. Each test confirms: (1) no exception is raised, (2) the predicted price is within the plausible ₹1,500–₹80,000 range, and (3) the output type is a Python float (not a numpy array). These tests must pass on every model update before the new model is promoted to production — forming the automated regression test suite for the ML system.

In [ ]:
def run_unit_tests():

    test_cases = [

        # Normal case
        {
        "duration_hours":3,
        "journey_month":5,
        "journey_day":12,
        "journey_weekday":2,
        "dep_hour":10,
        "airline_mean_price":9000,
        "source_freq":1200
        },

        # Very short flight
        {
        "duration_hours":0.5,
        "journey_month":3,
        "journey_day":10,
        "journey_weekday":4,
        "dep_hour":6,
        "airline_mean_price":5000,
        "source_freq":800
        },

        # Very long flight
        {
        "duration_hours":20,
        "journey_month":7,
        "journey_day":20,
        "journey_weekday":1,
        "dep_hour":22,
        "airline_mean_price":15000,
        "source_freq":2000
        },

        # Extreme airline pricing
        {
        "duration_hours":5,
        "journey_month":12,
        "journey_day":25,
        "journey_weekday":5,
        "dep_hour":14,
        "airline_mean_price":100000,
        "source_freq":1500
        },

        # Negative input test
        {
        "duration_hours":-2,
        "journey_month":8,
        "journey_day":5,
        "journey_weekday":3,
        "dep_hour":11,
        "airline_mean_price":7000,
        "source_freq":1000
        }

    ]

    print("Running unit tests...\n")

    for i, case in enumerate(test_cases):

        try:

            prediction = predict_flight_price(case)

            print(f"Test Case {i+1} SUCCESS")
            print("Input:", case)
            print("Predicted Price:", prediction)
            print("-"*40)

        except Exception as e:

            print(f"Test Case {i+1} FAILED")
            print("Input:", case)
            print("Error:", e)
            print("-"*40)

In [ ]:
if __name__ == "__main__":

    run_unit_tests()

Running unit tests...

Test Case 1 SUCCESS
Input: {'duration_hours': 3, 'journey_month': 5, 'journey_day': 12, 'journey_weekday': 2, 'dep_hour': 10, 'airline_mean_price': 9000, 'source_freq': 1200}
Predicted Price: 4792.611050650464
----------------------------------------
Test Case 2 SUCCESS
Input: {'duration_hours': 0.5, 'journey_month': 3, 'journey_day': 10, 'journey_weekday': 4, 'dep_hour': 6, 'airline_mean_price': 5000, 'source_freq': 800}
Predicted Price: 4456.470200160411
----------------------------------------
Test Case 3 SUCCESS
Input: {'duration_hours': 20, 'journey_month': 7, 'journey_day': 20, 'journey_weekday': 1, 'dep_hour': 22, 'airline_mean_price': 15000, 'source_freq': 2000}
Predicted Price: 4560.8901710974515
----------------------------------------
Test Case 4 SUCCESS
Input: {'duration_hours': 5, 'journey_month': 12, 'journey_day': 25, 'journey_weekday': 5, 'dep_hour': 14, 'airline_mean_price': 100000, 'source_freq': 1500}
Predicted Price: 4991.706255604797
--------

---
##  Prediction Engine — Module Complete

This module (`prediction_engine13.py`) is the **production prediction contract** for AirFair-Vista:

| Function | Input | Output |
|---|---|---|
| `engineer_features(user_input)` | Raw user dict | Engineered feature dict |
| `preprocess_input(feature_dict)` | Feature dict | Aligned DataFrame |
| `predict_flight_price(user_input)` | Raw user dict | Price in INR (float) |
| `run_unit_tests()` | None | Pass/fail for 5 edge cases |